# Model summaries

Prints `summary()` for each CoNNGaFit network and checks it against the model itself: every reported shape against a real forward pass on random data, and the per-block parameter counts against the model's total. The two regression networks are built and checked once per head option (`"conv"` and `"fc"`), including that both heads return the same output format.

| Section | Model |
|---|---|
| 1 | `Unet3d_ResBlock.NeuralNetwork` |
| 2 | `Unet3d_Dilated_ResNet.NeuralNetwork` |
| 3 | `Unet3d_DDPM.GaussianDiffusion` + `DenoisingUNet3d` |

Each section has a settings cell: edit it and re-run that section. Run the **Setup** cell first.

## Setup

In [1]:
import os
import sys
import torch

# Make the repo root importable whether the notebook is opened from the repo root or from examples/.
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(REPO_ROOT, "Unet_blocks.py")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Could not find the CoNNGaFit repo root (Unet_blocks.py) above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

SHOW_MODULES = False   # True also prints the full PyTorch module listing at the end of each summary
SHOW_LAYERS = False     # list each block's layers and their parameter counts under the block's row
SEED = 0               # random inputs/weights are seeded so re-runs give identical output
results = {}           # section name -> True/False, collected for the final check

In [2]:
def summary_printer(show_modules=SHOW_MODULES):
    """print_fn for model.summary(). With show_modules=False the long module printout (the only
    multi-line string summary() emits) is dropped, along with the duplicate rule line it leaves."""
    last = [None]

    def _print(s):
        if not show_modules and "\n" in s:
            return
        if s == last[0] and set(s) == {"="}:
            return
        last[0] = s
        print(s)
    return _print


def record_shapes(spec, run):
    """Run `run()` once under torch.no_grad() and return the (channels, dims) seen at each hook.

    spec: list aligned with stage_shapes() rows; each entry is (module, 'in' | 'out') to record a
    module's first input or its output, or None for a row with no module of its own (e.g. a
    functional resize), which is returned as None and skipped by compare_shapes.
    """
    shapes = [None] * len(spec)
    handles = []
    for k, entry in enumerate(spec):
        if entry is None:
            continue
        module, where = entry
        if where == "out":
            handles.append(module.register_forward_hook(
                lambda m, i, o, k=k: shapes.__setitem__(k, o.shape)))
        else:
            handles.append(module.register_forward_pre_hook(
                lambda m, i, k=k: shapes.__setitem__(k, i[0].shape)))
    try:
        with torch.no_grad():
            run()
    finally:
        for h in handles:
            h.remove()
    return [None if s is None else (int(s[1]), tuple(int(v) for v in s[2:])) for s in shapes]


def compare_shapes(rows, actual):
    """Print predicted vs actual (features, dims) for every stage_shapes() row; return True if all
    checked rows match."""
    w = max(len(label) for label, _, _ in rows)
    fmt = lambda f, d: f"{f:>4} x {' x '.join(str(v) for v in d)}"
    print(f"{'stage':<{w}}   {'predicted':<24} {'actual':<24} result")
    all_ok = True
    for (label, feats, dims), act in zip(rows, actual):
        pred = fmt(feats, dims)
        if act is None:
            print(f"{label:<{w}}   {pred:<24} {'(no module)':<24} not checked")
            continue
        ok = (feats, tuple(dims)) == act
        all_ok &= ok
        print(f"{label:<{w}}   {pred:<24} {fmt(*act):<24} {'OK' if ok else 'MISMATCH'}")
    print(f"\n{'All shapes match.' if all_ok else 'Shape MISMATCH - summary() disagrees with forward().'}")
    return all_ok


def head_shape_spec(model):
    """(module, 'in'|'out') per head row of stage_shapes(): conv head rows or the fc head's pool."""
    if model.head_type == "conv":
        h = model.conv_head
        return [(h.pool, "out"), None] + [(b, "out") for b in h.blocks] + [(h.out, "out")]
    return [(model.avgpool0, "out")]


def check_param_counts(breakdown, model):
    """True if the per-block counts summary() reports add up to the model's total parameter count."""
    listed = sum(n for _, n in breakdown)
    total = sum(p.numel() for p in model.parameters())
    print(f"parameters listed per block: {listed:,}   model total: {total:,}   "
          f"{'OK' if listed == total else 'MISMATCH'}")
    return listed == total


def check_heads(name, build, heads, shape_spec, x):
    """Build a regression network once per head option and check each one: summary() shapes against
    a forward pass, per-block parameter counts against the total, and the output format
    (nBatch, output_size). With several heads, also check they all return identically shaped
    outputs, then compare their sizes. Records every check in `results`; returns {head: model}."""
    models, out_shapes = {}, {}
    for head in heads:
        print(f"\n{'#' * 24}  {name}, head={head!r}  {'#' * 24}")
        torch.manual_seed(SEED)
        model = build(head).eval()
        model.summary(print_fn=summary_printer(), layers=SHOW_LAYERS)
        print()
        outputs = []
        actual = record_shapes(shape_spec(model), lambda: outputs.append(model(x)))
        shapes_ok = compare_shapes(model.stage_shapes(), actual)
        params_ok = check_param_counts(model.param_breakdown(), model)
        out = outputs[0] if isinstance(outputs[0], tuple) else (outputs[0],)   # (mean,) or (mean, logvar)
        out_shapes[head] = tuple(tuple(o.shape) for o in out)
        expected = (x.shape[0], model.output_size)
        format_ok = all(s == expected for s in out_shapes[head])
        print(f"output shape(s): {out_shapes[head]}   expected (nBatch, output_size) = {expected}   "
              f"{'OK' if format_ok else 'MISMATCH'}")
        results[f"{name}, head={head!r}"] = shapes_ok and params_ok and format_ok
        models[head] = model

    if len(heads) > 1:
        same = len(set(out_shapes.values())) == 1
        print(f"\n{name}: every head returns the same output shape(s): {'OK' if same else 'MISMATCH'}")
        results[f"{name}, heads interchangeable"] = same
        print(f"{'head':<6} {'total params':>14} {'head params':>14}")
        for head, model in models.items():
            total = sum(p.numel() for p in model.parameters())
            in_head = sum(n for label, n in model.param_breakdown() if label.startswith(("head", "fc")))
            print(f"{head!r:<6} {total:>14,} {in_head:>14,}")
    return models

## 1. Residual U-Net (`Unet3d_ResBlock`)

Settings follow the `NeuralNetwork` constructor, except `head`: the network is built and checked once for every entry in `resblock_heads`. `"conv"` is the convolutional head (size independent of the cube and map); `"fc"` is the pooled fully-connected head, for which `pool_output` / `nFC` / `n_fc_layers` apply (`nFC` is an int, 0 = no hidden FC layers, or a list of widths).

In [3]:
from Unet3d_ResBlock import NeuralNetwork as ResBlockNet

resblock_cfg = dict(
    input_shape=(128, 128, 128),     # (nSpec, nX, nY)
    nFilt0=8,
    kernel_stem=(3, 3, 3),
    kernel_coder=(3, 3, 3),
    nFC=0,
    n_stages=4,
    upsample_mode="conv_transpose",  # or "trilinear"
    predict_variance=True,
    n_head_blocks=2,                 # 2-d residual blocks in the conv head
)
resblock_heads = ("conv", "fc")      # every head listed is built and checked

In [4]:
def resblock_shape_spec(model):
    """(module, 'in'|'out') per stage_shapes() row, in the same order."""
    spec = [(model.layer_stem, "in"), (model.layer_stem, "out")]
    spec += [(stage, "out") for stage in model.encoder]
    for up, dc in zip(model.upsamples, model.dc_layers):
        spec += [(up, "out"), (dc, "in"), (dc, "out")]   # upsample, + skip (concat), deconv block
    return spec + head_shape_spec(model)

In [5]:
resblock_models = check_heads(
    "ResBlock U-Net",
    build=lambda head: ResBlockNet(**resblock_cfg, head=head),
    heads=resblock_heads,
    shape_spec=resblock_shape_spec,
    x=torch.randn(1, *resblock_cfg["input_shape"]),
)


########################  ResBlock U-Net, head='conv'  ########################
output size= 16384
NeuralNetwork  |  4 encoder stages, 3 decoder stages  |  kernel_stem=(3, 3, 3)  |  kernel_coder=(3, 3, 3)  |  upsample_mode='conv_transpose'  |  predict_variance=True  |  head='conv'
input:  (nBatch, nSpec=128, nX=128, nY=128)  ->  output: (nBatch, 16384)
activations: stem=LeakyReLU  encoder/decoder blocks=ReLU  head blocks=ReLU  output=Identity
-----------------------------------------------------------------------------------------
  input datacube                    features=1      params=-           dims=128 x 128 x 128
  stem (conv + maxpool)             features=8      params=224         dims=32 x 32 x 32 activation=LeakyReLU
  encoder stage 0 (no downsample)   features=8      params=7,008       dims=32 x 32 x 32 activation=ReLU
  encoder stage 1                   features=16     params=24,560      dims=16 x 16 x 16 activation=ReLU
  encoder stage 2                   features=32   

## 2. Dilated residual network (`Unet3d_Dilated_ResNet`)

`stem_stride` > 1 downsamples once in the Encoder (every axis to ceil(N/stride)); the rest of the network keeps that shape. As in section 1, the network is built and checked once for every entry in `dilated_heads`.

In [6]:
from Unet3d_Dilated_ResNet import NeuralNetwork as DilatedNet

dilated_cfg = dict(
    input_shape=(128, 128, 128),     # (nSpec, nX, nY)
    nFilt0=8,
    kernel_stem=(3, 3, 3),           # Encoder / Decoder kernel
    kernel_coder=(3, 3, 3),          # dilated conv kernel
    nFC=0,
    n_blocks=4,
    dilation_rates=(1, 2, 4, 8, 4, 2, 1),
    stem_stride=(1, 1, 1),
    predict_variance=False,
    n_head_blocks=2,                 # 2-d residual blocks in the conv head
)
dilated_heads = ("conv", "fc")       # every head listed is built and checked

In [7]:
def dilated_shape_spec(model):
    """(module, 'in'|'out') per stage_shapes() row, in the same order."""
    spec = [(model.layer_encode, "in"), (model.layer_encode, "out")]
    spec += [(block, "out") for block in model.process]
    spec.append((model.layer_decode, "out"))
    return spec + head_shape_spec(model)

In [8]:
dilated_models = check_heads(
    "Dilated ResNet",
    build=lambda head: DilatedNet(**dilated_cfg, head=head),
    heads=dilated_heads,
    shape_spec=dilated_shape_spec,
    x=torch.randn(1, *dilated_cfg["input_shape"]),
)


########################  Dilated ResNet, head='conv'  ########################
output size= 16384
NeuralNetwork  |  4 process blocks  |  dilation_rates=(1, 2, 4, 8, 4, 2, 1)  |  kernel_stem=(3, 3, 3)  |  kernel_coder=(3, 3, 3)  |  stem_stride=(1, 1, 1)  |  predict_variance=False  |  head='conv'
input:  (nBatch, nSpec=128, nX=128, nY=128)  ->  output: (nBatch, 16384)
activations: encode=Identity  process blocks=ReLU  decode=Identity  head blocks=ReLU  output=Identity
---------------------------------------------------------------------------------------
  input datacube                  features=1      params=-           dims=128 x 128 x 128
  encode                          features=8      params=224         dims=128 x 128 x 128 activation=Identity  stride=(1, 1, 1)
  process block 0                 features=8      params=12,264      dims=128 x 128 x 128 activation=ReLU  dilation_rates=(1, 2, 4, 8, 4, 2, 1)
  process block 1                 features=8      params=12,264      dims=128

## 3. Conditional diffusion model (`Unet3d_DDPM`)

The DDPM is fully convolutional, so the cube and map sizes are chosen here rather than in the model. `map_shape=None` uses the cube's (nX, nY).

For the 256³ setup use `input_shape=(256, 256, 256)`, `n_stages=5`, `nFilt0=16` (one forward pass takes a few seconds on CPU).

In [9]:
from Unet3d_DDPM import DenoisingUNet3d, GaussianDiffusion

ddpm_data = dict(
    input_shape=(128, 128, 128),     # (nSpec, nX, nY)
    map_shape=None,                  # (H, W) or None for (nX, nY)
)
ddpm_net_cfg = dict(
    nFilt0=8,
    kernel_stem=(3, 3, 3),
    kernel_coder=(3, 3, 3),
    n_stages=4,
    n_head_blocks=2,
    upsample_mode="trilinear",       # or "conv_transpose"
)
ddpm_diffusion_cfg = dict(
    n_timesteps=1000,
    schedule="cosine",               # or "linear"
)

In [10]:
def ddpm_shape_spec(net):
    """(module, 'in'|'out') per DenoisingUNet3d.stage_shapes() row, in the same order."""
    spec = [(net.stem, "in"), (net.stem, "out")]          # input (cube + tiled map), stem
    spec += [(stage[1], "out") for stage in net.encoder]
    for up, blocks in zip(net.upsamples, net.decoder):
        spec += [(up, "out"), (blocks[0], "in"), (blocks[1], "out")]   # upsample, + skip, deconv
    spec += [(net.head.pool, "out"),
             None,                                     # bilinear resize: functional, no module
             (net.head.blocks[0], "in")]               # + noisy map y_t (concat)
    spec += [(block, "out") for block in net.head.blocks]
    spec.append((net.head.out, "out"))
    return spec


input_shape = ddpm_data["input_shape"]
map_shape = ddpm_data["map_shape"] or tuple(input_shape[1:])

torch.manual_seed(SEED)
denoiser = DenoisingUNet3d(**ddpm_net_cfg).eval()
diffusion = GaussianDiffusion(denoiser, target_shape=map_shape, **ddpm_diffusion_cfg)
diffusion.summary(input_shape, print_fn=summary_printer(), layers=SHOW_LAYERS)

GaussianDiffusion  |  T=1000  |  schedule='cosine'  |  target_shape=(128, 128)  |  clip_range=(-1.0, 1.0)
signal fraction sqrt(alpha_bar): 1.0000 at t=0 -> 4.93e-05 at t=999
training: MSE(eps_theta(cube, y_t, t), eps) with t ~ U{0..T-1};  sampling: DDIM update (eta=1 with all T steps = DDPM)
map: (nBatch, 128, 128), labels flattened to (nBatch, 16384)
DenoisingUNet3d  |  4 encoder stages, 3 decoder stages  |  kernel_stem=(3, 3, 3)  |  kernel_coder=(3, 3, 3)  |  upsample_mode='trilinear'  |  t_dim=32  |  GroupNorm max_groups=8
input:  cube (nBatch, nSpec=128, nX=128, nY=128) + noisy map (nBatch, 128, 128) + t (nBatch,)  ->  output: eps (nBatch, 128, 128)
timestep embedding: t -> 8 sinusoidal features -> MLP (SiLU) -> 32; FiLM scale/shift into every encoder, decoder and head block
activations: stem=LeakyReLU  encoder/decoder blocks=SiLU  head blocks=SiLU
-----------------------------------------------------------------------------------------
  timestep embedding                features=

In [11]:
cube = torch.randn(1, *input_shape)
y_t = torch.randn(1, *map_shape)
t = torch.randint(0, ddpm_diffusion_cfg["n_timesteps"], (1,))
actual = record_shapes(ddpm_shape_spec(denoiser), lambda: denoiser(cube, y_t, t))
shapes_ok = compare_shapes(denoiser.stage_shapes(input_shape, map_shape), actual)
results["DDPM denoiser"] = check_param_counts(denoiser.param_breakdown(input_shape, map_shape), denoiser) and shapes_ok

stage                             predicted                actual                   result
input: cube + tiled noisy map        2 x 128 x 128 x 128      2 x 128 x 128 x 128   OK
stem (conv + maxpool)                8 x 32 x 32 x 32         8 x 32 x 32 x 32      OK
encoder stage 0 (no downsample)      8 x 32 x 32 x 32         8 x 32 x 32 x 32      OK
encoder stage 1                     16 x 16 x 16 x 16        16 x 16 x 16 x 16      OK
encoder stage 2                     32 x 8 x 8 x 8           32 x 8 x 8 x 8         OK
encoder stage 3                     64 x 4 x 4 x 4           64 x 4 x 4 x 4         OK
decoder 0: upsample                 32 x 8 x 8 x 8           32 x 8 x 8 x 8         OK
decoder 0: + skip(enc 2)            64 x 8 x 8 x 8           64 x 8 x 8 x 8         OK
decoder 0: deconv block             32 x 8 x 8 x 8           32 x 8 x 8 x 8         OK
decoder 1: upsample                 16 x 16 x 16 x 16        16 x 16 x 16 x 16      OK
decoder 1: + skip(enc 1)            32 

## Result

One line per check: shapes + parameter counts + output format for each model and head, and whether the heads of each regression network are interchangeable.

In [12]:
w = max(len(name) for name in results)
for name, ok in results.items():
    print(f"{name:<{w}}  {'OK' if ok else 'MISMATCH'}")
assert all(results.values()), "at least one summary() or head check disagrees with its model"

ResBlock U-Net, head='conv'            OK
ResBlock U-Net, head='fc'              OK
ResBlock U-Net, heads interchangeable  OK
Dilated ResNet, head='conv'            OK
Dilated ResNet, head='fc'              OK
Dilated ResNet, heads interchangeable  OK
DDPM denoiser                          OK
